## 2.1 Chuẩn hóa lưới 1 phút
- Đầu vào: `data/raw_numerics/*.parquet` (chỉ record `status == "ok"` trong manifest).
- Đầu ra: `data/clean_1min/<record>.parquet`: mỗi dòng là 1 phút (`spo2_mean`, `spo2_min`, `spo2_valid`, `pulse_mean`, `hr_mean`, `resp_mean`, `abp_mean`).
- Giá trị ngoài khoảng sinh lý bị đổi thành NaN; mỗi record tối đa 72 giờ.
- Có resume: record nào đã có file đầu ra thì bỏ qua. Chạy lại cell khi có thêm dữ liệu mới tải về.

In [1]:
import os, numpy as np, pandas as pd
from tqdm.auto import tqdm

RAW_DIR, CLEAN_DIR = "data/raw_numerics", "data/clean_1min"
MANIFEST_PATH = "data/download_manifest.parquet"
CAP_HOURS = 72
RANGES = {"SPO2": (50, 100), "PULSE": (30, 220), "HR": (30, 220), "RESP": (3, 60), "ABPMEAN": (20, 250)}
os.makedirs(CLEAN_DIR, exist_ok=True)

def to_1min(df, fs):
    spm = max(1, int(round(fs * 60)))                 
    df = df.iloc[: int(CAP_HOURS * 60 * spm)].copy()
    for c, (lo, hi) in RANGES.items():
        df[c] = df[c].where((df[c] >= lo) & (df[c] <= hi))
    g = np.arange(len(df)) // spm
    out = df.groupby(g).mean().astype("float32")
    out.columns = [c.lower().replace("abpmean", "abp") + "_mean" for c in out.columns]
    out["spo2_min"] = df["SPO2"].groupby(g).min().astype("float32")
    out["spo2_valid"] = df["SPO2"].notna().groupby(g).mean().astype("float32")
    return out.rename_axis("minute").reset_index()

m = pd.read_parquet(MANIFEST_PATH)
ok = m[m["status"] == "ok"]
todo = [(r, fs) for r, fs in zip(ok["record"], ok["fs"])
        if not os.path.exists(f"{CLEAN_DIR}/{r.split('/')[-1]}.parquet")]
print(f"Record ok: {len(ok)} | đã chuẩn hóa: {len(ok) - len(todo)} | còn: {len(todo)}")

for rec, fs in tqdm(todo, desc="1-min grid"):
    name = rec.split("/")[-1]
    out = to_1min(pd.read_parquet(f"{RAW_DIR}/{name}.parquet"), fs)
    out.to_parquet(f"{CLEAN_DIR}/{name}.tmp", index=False)
    os.replace(f"{CLEAN_DIR}/{name}.tmp", f"{CLEAN_DIR}/{name}.parquet")   

Record ok: 1426 | đã chuẩn hóa: 1426 | còn: 0


1-min grid: 0it [00:00, ?it/s]

In [2]:
ok = pd.read_parquet(MANIFEST_PATH).query("status == 'ok'").copy()
ok["loai_fs"] = np.where(ok["fs"] > 0.5, "1 Hz", "1 mau/phut")
print(ok.groupby("loai_fs").agg(records=("record", "size"), patients=("patient_id", "nunique"),
                                gio=("hours", "sum")).round(0), "\n")

def runs(mask):
    d = np.diff(np.r_[0, mask.astype(int), 0])
    return np.where(d == -1)[0] - np.where(d == 1)[0]     # độ dài các đoạn liên tiếp

res, tot_min = {}, 0
for r, pid in zip(ok["record"], ok["patient_id"]):
    d = pd.read_parquet(f"{CLEAN_DIR}/{r.split('/')[-1]}.parquet", columns=["spo2_mean"])
    tot_min += d["spo2_mean"].notna().sum()
    for thr in (88, 90, 92):
        rl = runs((d["spo2_mean"] < thr).values)
        for k in (1, 2, 3, 5):
            n = (rl >= k).sum()
            e = res.setdefault((thr, k), [0, set()]); e[0] += n
            if n: e[1].add(pid)

print(f"Giờ SpO2 hợp lệ: {tot_min / 60:,.0f} | Bệnh nhân: {ok['patient_id'].nunique()}")
print("Số sự kiện / số bệnh nhân có ít nhất 1 sự kiện (sk / BN):")
display(pd.DataFrame({f"<{t}%": {f">={k}ph": f"{res[(t, k)][0]} sk / {len(res[(t, k)][1])} BN"
                                 for k in (1, 2, 3, 5)} for t in (88, 90, 92)}))

            records  patients      gio
loai_fs                               
1 Hz            983       566  34298.0
1 mau/phut      443       244  17600.0 

Giờ SpO2 hợp lệ: 45,347 | Bệnh nhân: 805
Số sự kiện / số bệnh nhân có ít nhất 1 sự kiện (sk / BN):


,<88%,<90%,<92%
>=1ph,11810 sk / 626 BN,17867 sk / 674 BN,31280 sk / 731 BN
>=2ph,5503 sk / 496 BN,8988 sk / 582 BN,16770 sk / 665 BN
>=3ph,3143 sk / 404 BN,5318 sk / 488 BN,10770 sk / 597 BN
>=5ph,1455 sk / 278 BN,2673 sk / 375 BN,5929 sk / 493 BN
